# Rung 40 — thinking at INFERENCE, on 1/10 of the eval

**THE ONE VARIABLE: `enable_thinking` False → True.** Same checkpoint, same data, same
inference path, same 626 questions. The control is the arm's OWN archived no-thinking
run, so no reference is re-executed and none can drift.

Two of the three requested modes are not runnable arms and that is measured, not assumed:
`preserve_thinking` emits a **byte-identical** prompt to `enable_thinking` on a
single-turn conversation (FRAME has no previous turn to preserve), and A2 has **no
thinking mode at all** — zero `enable_thinking` in its chat template.

⚠️ **Pre-registered confound.** This checkpoint was trained 901 steps on the no-thinking
path and **zero** on the thinking path. A null cannot separate *"reasoning does not help
FRAME"* from *"our SFT overwrote the reasoning ability"*. It is still actionable for the
submission — which ships this checkpoint — but it does not generalise.

## 1 — Parameters (papermill overrides this cell)

In [ ]:
MERGED_DIR = ""          # the arm's merged/ — set by the chain
RUN_NAME   = "40_think_probe"
WORK_DIR   = "/workspace/repo_leo/experiments/40-gen36-recipe-connector/runs"
HF_HOME    = "/workspace/hf_cache"

N_SUBSET   = 625         # ~1/10 of 6252; stratified, NEVER head-of-list
SEED       = 42

# 🔴 64 is the production cap and cannot hold a reasoning trace — that is exactly what
# scored the rung 23a smoke 0/24. Only the thinking arm needs this; the no-thinking
# reference keeps its own archived answers at its own budget, which is the budget each
# mode needs to function at all.
MAX_NEW_TOKENS = 512

## 2 — Environment. Fail here, not after loading a 27B.

In [ ]:
import os, sys, logging
from pathlib import Path

os.environ["HF_HOME"] = HF_HOME
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
sys.path.insert(0, str(Path.cwd() / "_tools"))

import torch
assert torch.cuda.is_available(), "no CUDA"
assert MERGED_DIR and Path(MERGED_DIR).is_dir(), f"no merged checkpoint at {MERGED_DIR!r}"
print("gpu     ", torch.cuda.get_device_name(0))
print("free GiB", round(torch.cuda.mem_get_info()[0] / 2**30, 1))
print("merged  ", MERGED_DIR)

## 3 — Config. Inline, per the repo spec.

In [ ]:
from thinking_probe import ProbeConfig, stratified_qids, run_thinking_arm, score_subset

cfg = ProbeConfig(
    merged_dir=MERGED_DIR,
    out_dir=str(Path(WORK_DIR) / RUN_NAME / "eval"),
    run_name=RUN_NAME,
    n_subset=N_SUBSET,
    seed=SEED,
    max_new_tokens_thinking=MAX_NEW_TOKENS,
)
cfg

## 4 — The subset. The SAME qIDs every arm answers.

In [ ]:
qids = stratified_qids(cfg)
print(f"{len(qids)} qIDs, stratified over (answer_format x ID/OOD), seed {SEED}")

## 5 — The one arm that needs a GPU: this checkpoint, thinking ON.

In [ ]:
report = run_thinking_arm(cfg, qids)
print("proxy_leaderboard:", report.get("proxy_leaderboard"))
print("bucket_mean      :", report.get("bucket_mean"))
print("n_timed_out      :", report.get("n_timed_out"))

## 6 — The references, subset to the same qIDs. Zero GPU.

In [ ]:
import json

rows = [(f"{RUN_NAME} (THINKING)", report)]
for name, path in cfg.reference_csvs.items():
    if not Path(path).exists():
        print(f"  skip {name}: {path} not on this volume")
        continue
    rows.append((name, score_subset(path, set(qids))))

print(f"\n{'arm':<26}{'proxy':>9}{'bucket':>9}{'agg_ID':>9}{'obj_ID':>9}")
print("-" * 62)
for name, r in rows:
    print(f"{name:<26}{r['proxy_leaderboard']:>9.4f}{r['bucket_mean']:>9.4f}"
          f"{r['aggregation_ID']:>9.4f}{r['object_recognition_ID']:>9.4f}")

out = Path(cfg.out_dir) / "RESULTS_thinking_probe.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(
    {"n_subset": len(qids), "seed": SEED, "max_new_tokens": MAX_NEW_TOKENS,
     "qids": qids,
     "arms": {n: {k: r.get(k) for k in
                  ("proxy_leaderboard", "bucket_mean", "aggregation_ID",
                   "object_recognition_ID", "acc_ID", "acc_OOD", "n_timed_out")}
              for n, r in rows}}, indent=2, default=str))
print("\nwritten", out)